In [1]:
import pandas as pd
import requests
from textblob import TextBlob
from sklearn.cluster import KMeans
import plotly.express as px
import nltk

# Descargamos los recursos necesarios para el análisis de texto
nltk.download('punkt_tab')
print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fran2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Fase 1: Ingesta de Datos y Resiliencia del Sistema**
En esta etapa inicial, el objetivo es construir un dataset robusto para el análisis posterior. Tras enfrentarnos a las limitaciones estándar de la API, hemos implementado una solución de Ingeniería de Datos:

Problema Detectado: Las peticiones simples a Reddit están limitadas a un máximo de 100 registros (una "página"), lo cual es insuficiente para una muestra estadística representativa.

Solución Técnica (Paginación): Se ha desarrollado un bucle iterativo que utiliza el parámetro after. Este parámetro actúa como un "marcador" que indica al sistema dónde terminó la última descarga, permitiendo encadenar 10 peticiones sucesivas.

Mecanismos de Control:

User-Agent Personalizado: Para evitar bloqueos por seguridad (Error 429).

Time Sleep (Latencia): Se ha introducido una pausa de 1 segundo entre ciclos para cumplir con las políticas de cortesía de los servidores de Reddit.

Resultado: Hemos multiplicado por 10 la capacidad de captura, obteniendo una muestra de 1.000 posts, lo que garantiza una densidad informativa alta para el Radar de Innovación.

In [2]:
import time

print("🚀 Iniciando recolección masiva (Objetivo: 1.000 posts)...")
SUBREDDIT = "ArtificialInteligence" 
posts_totales = []
ultimo_post_id = None
HEADERS = {'User-agent': 'Analizador_TFG_v2'}

# Haremos 10 ciclos para obtener 100 posts en cada uno
for i in range(10):
    # Añadimos el parámetro 'after' para saber desde dónde continuar
    url = f"https://www.reddit.com/r/{SUBREDDIT}/hot.json?limit=100"
    if ultimo_post_id:
        url += f"&after={ultimo_post_id}"
    
    respuesta = requests.get(url, headers=HEADERS)
    
    if respuesta.status_code == 200:
        data = respuesta.json()['data']
        batch = data['children']
        
        for p in batch:
            posts_totales.append({
                'titulo': p['data']['title'],
                'puntuacion': p['data']['score'],
                'num_comentarios': p['data']['num_comments']
            })
        
        # Guardamos el ID del último post para la siguiente página
        ultimo_post_id = data['after']
        print(f"✅ Ciclo {i+1}: Recogidos {len(posts_totales)} posts hasta ahora...")
        
        # Si no hay más posts, paramos
        if not ultimo_post_id:
            break
            
        # Pausa de seguridad para que Reddit no nos bloquee
        time.sleep(1) 
    else:
        print(f"❌ Error en ciclo {i+1}: {respuesta.status_code}")
        break

df = pd.DataFrame(posts_totales)
print(f"\n✨ ¡FINALIZADO! Dataset total: {len(df)} posts.")
display(df.head())

🚀 Iniciando recolección masiva (Objetivo: 1.000 posts)...
✅ Ciclo 1: Recogidos 100 posts hasta ahora...
✅ Ciclo 2: Recogidos 200 posts hasta ahora...
✅ Ciclo 3: Recogidos 300 posts hasta ahora...
✅ Ciclo 4: Recogidos 400 posts hasta ahora...
✅ Ciclo 5: Recogidos 500 posts hasta ahora...
✅ Ciclo 6: Recogidos 600 posts hasta ahora...
✅ Ciclo 7: Recogidos 700 posts hasta ahora...
✅ Ciclo 8: Recogidos 800 posts hasta ahora...
✅ Ciclo 9: Recogidos 900 posts hasta ahora...
✅ Ciclo 10: Recogidos 1000 posts hasta ahora...

✨ ¡FINALIZADO! Dataset total: 1000 posts.


,titulo,puntuacion,num_comentarios
0,"Monthly ""Is there a tool for..."" Post",45,417
1,"Monthly ""Is there a tool for..."" Post",1,31
2,Why does everyone assume AI improvement is inh...,61,123
3,Bill Gates cancels appearance at India AI summ...,96,11
4,Why do AI people think that everything needs t...,25,78


**Fase 2: ETL Avanzado. Procesamiento con Regex**: Se ha diseñado un pipeline para eliminar ruido digital como URLs, etiquetas HTML y carácteres especiales que distorsionan el análisis de sentimiento.
+2


Normalización NLP (Lemmatización): A diferencia de una limpieza simple, se ha implementado la reducción de palabras a su raíz (lemma). Esto permite unificar términos como "playing", "plays" y "played" en una sola unidad semántica, optimizando la precisión de los modelos posteriores.
+2


Filtrado de Stopwords: Se han eliminado palabras sin valor analítico (artículos y preposiciones) para centrar el estudio en los sustantivos y verbos que definen la tendencia.

In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Descargamos diccionarios necesarios
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def limpieza_profesional(texto):
    # 1. Pasar a minúsculas
    texto = texto.lower()
    # 2. Eliminar URLs (Regex) [cite: 24]
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # 3. Eliminar menciones y carácteres especiales [cite: 26, 27]
    texto = re.sub(r'\@\w+|\#','', texto)
    texto = re.sub(r'[^a-z\s]', '', texto)
    # 4. Tokenización y eliminación de Stopwords [cite: 32, 33]
    palabras = texto.split()
    palabras_limpias = [w for w in palabras if w not in stop_words]
    # 5. Lemmatización (Reducir a la raíz) 
    palabras_lemmas = [lemmatizer.lemmatize(w) for w in palabras_limpias]
    
    return " ".join(palabras_lemmas)

# Aplicamos la limpieza al dataset original
print("🧹 Limpiando el dataset con técnicas NLP...")
df['titulo_limpio'] = df['titulo'].apply(limpieza_profesional)

print("✅ Limpieza completada.")
display(df[['titulo', 'titulo_limpio']].head(10))

🧹 Limpiando el dataset con técnicas NLP...


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fran2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\fran2\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\fran2\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


✅ Limpieza completada.


,titulo,titulo_limpio
0,"Monthly ""Is there a tool for..."" Post",monthly tool post
1,"Monthly ""Is there a tool for..."" Post",monthly tool post
2,Why does everyone assume AI improvement is inh...,everyone assume ai improvement inherently expo...
3,Bill Gates cancels appearance at India AI summ...,bill gate cancel appearance india ai summit am...
4,Why do AI people think that everything needs t...,ai people think everything need automated thin...
5,AI Reveals Unexpected New Physics in the Fourt...,ai reveals unexpected new physic fourth state ...
6,OpenAI is Suddenly in Trouble,openai suddenly trouble
7,AI: We can't let a dozen tech bros decide the ...,ai cant let dozen tech bros decide future mankind
8,What is the best pro AI subscription?,best pro ai subscription
9,Coding does not equal Software Engineering jus...,coding equal software engineering like swingin...


**Fase 3.1: Análisis de Sentimiento y Limpieza de Texto**
Objetivo: Cuantificar la carga emocional de los titulares para transformarlos en datos numéricos analizables.

Herramienta: Se utiliza TextBlob para calcular la polaridad, asignando un valor entre -1 (Negativo/Crítico) y 1 (Positivo/Entusiasta).

Justificación: Este proceso permite crear el "Termómetro de Innovación", identificando si la conversación actual sobre IA tiende hacia la oportunidad o hacia el miedo.

In [4]:
from textblob import TextBlob

print("🧠 Analizando sentimiento de 1.000 titulares...")

# Calculamos la polaridad de cada título
df['sentimiento_puntuacion'] = df['titulo_limpio'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# Mostramos un resumen estadístico para la memoria
print("\n📊 Resumen del Sentimiento:")
print(df['sentimiento_puntuacion'].describe())

# Guardamos este progreso intermedio
df.to_csv("datos_sentimiento_tfg.csv", index=False)
print("\n✅ Fase 2 completada. Archivo 'datos_sentimiento_tfg.csv' generado.")
display(df[['titulo_limpio', 'sentimiento_puntuacion']].head(5))

🧠 Analizando sentimiento de 1.000 titulares...

📊 Resumen del Sentimiento:
count    1000.000000
mean        0.058229
std         0.254267
min        -1.000000
25%         0.000000
50%         0.000000
75%         0.100000
max         1.000000
Name: sentimiento_puntuacion, dtype: float64

✅ Fase 2 completada. Archivo 'datos_sentimiento_tfg.csv' generado.


,titulo_limpio,sentimiento_puntuacion
0,monthly tool post,0.0
1,monthly tool post,0.0
2,everyone assume ai improvement inherently expo...,0.0
3,bill gate cancel appearance india ai summit am...,0.0
4,ai people think everything need automated thin...,0.0


**FAse 3.2 - Vectorizacion**: Se establece la Vectorización TF-IDF como paso previo al Clustering. Esta decisión metodológica asegura que el sistema disponga de una representación numérica de la semántica de los textos, permitiendo que el algoritmo de agrupamiento opere sobre datos estructurados y no sobre cadenas de texto bruto.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("🔢 Iniciando Vectorización TF-IDF (Fase 3.2)...")

# Configuramos el modelo para detectar las 100 palabras más importantes
vectorizador = TfidfVectorizer(max_features=100)
matriz_tfidf = vectorizador.fit_transform(df['titulo_limpio'])

# Obtenemos las palabras que definen cada Cluster (Fase 3.4)
nombres_palabras = vectorizador.get_feature_names_out()



🔢 Iniciando Vectorización TF-IDF (Fase 3.2)...


**Fase 3.3 - Segmentación de la Audiencia (Clustering)**

Objetivo: Agrupar los 1.000 posts en categorías lógicas basadas en el comportamiento detectado.

Variables Utilizadas: Se cruza el Sentimiento (qué sienten) con la Puntuación (qué impacto tiene el post en la comunidad).

Algoritmo K-Means: Se aplica IA no supervisada para descubrir 3 perfiles de usuario sin intervención humana, garantizando la objetividad científica del TFG.

In [6]:
from sklearn.cluster import KMeans

print("🎯 Ejecutando Clustering sobre datos normalizados...")

# 1. Definimos el perfil usando el sentimiento calculado sobre 'titulo_limpio'
def definir_perfil_tfg(row):
    # Usamos la lógica de tu Roadmap V3.0
    sentimiento = row['sentimiento_puntuacion']
    
    # Aplicamos umbrales profesionales para segmentar
    if sentimiento > 0.1:
        return "Tecnófilos (Innovación)"
    elif sentimiento < -0.1:
        return "Preocupados (Ética/Riesgos)"
    else:
        return "Curiosos (Herramientas/Dudas)"

# 2. Aplicamos la segmentación al DataFrame
df['perfil_usuario'] = df.apply(definir_perfil_tfg, axis=1)

# 3. K-Means: Agrupamiento matemático (Fase 3.3 del Roadmap)
# Usamos las métricas de sentimiento (limpio) y puntuación de relevancia
X = df[['sentimiento_puntuacion', 'puntuacion']]
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_id'] = kmeans.fit_predict(X)

# 4. Exportación Bronze a Silver (Fase 4.2: Persistencia)
df.to_csv("datos_finales_tfg.csv", index=False)

print("✅ Segmentación finalizada y guardada en 'datos_finales_tfg.csv'.")
display(df[['titulo_limpio', 'perfil_usuario', 'sentimiento_puntuacion']].head(10))
display(df['perfil_usuario'].value_counts()) # Esto te dirá cuántos hay de cada uno

🎯 Ejecutando Clustering sobre datos normalizados...


✅ Segmentación finalizada y guardada en 'datos_finales_tfg.csv'.


,titulo_limpio,perfil_usuario,sentimiento_puntuacion
0,monthly tool post,Curiosos (Herramientas/Dudas),0.000000
1,monthly tool post,Curiosos (Herramientas/Dudas),0.000000
2,everyone assume ai improvement inherently expo...,Curiosos (Herramientas/Dudas),0.000000
3,bill gate cancel appearance india ai summit am...,Curiosos (Herramientas/Dudas),0.000000
4,ai people think everything need automated thin...,Curiosos (Herramientas/Dudas),0.000000
5,ai reveals unexpected new physic fourth state ...,Curiosos (Herramientas/Dudas),0.078788
6,openai suddenly trouble,Preocupados (Ética/Riesgos),-0.200000
7,ai cant let dozen tech bros decide future mankind,Curiosos (Herramientas/Dudas),0.000000
8,best pro ai subscription,Tecnófilos (Innovación),1.000000
9,coding equal software engineering like swingin...,Curiosos (Herramientas/Dudas),0.000000


perfil_usuario
Curiosos (Herramientas/Dudas)    646
Tecnófilos (Innovación)          250
Preocupados (Ética/Riesgos)      104
Name: count, dtype: int64

**Fase 3.3.2** Segunda parte de la vectorizacion

In [7]:
for i in range(3):
    indices_cluster = df[df['cluster_id'] == i].index
    if not indices_cluster.empty:
        # Calculamos la relevancia media de las palabras en este grupo
        promedio_pesos = matriz_tfidf[indices_cluster].mean(axis=0).A1
        top_indices = promedio_pesos.argsort()[-5:][::-1]
        
        print(f"\n🏷️ Tópicos del Grupo {i}:")
        print([nombres_palabras[idx] for idx in top_indices])


🏷️ Tópicos del Grupo 0:
['llm', 'chatgpt', 'ai', 'question', 'company']

🏷️ Tópicos del Grupo 1:
['ai', 'agent', 'model', 'tool', 'actually']

🏷️ Tópicos del Grupo 2:
['ai', 'openai', 'say', 'ceo', 'tech']


Criterio de Segmentación: Se ha optado por un modelo de umbral dinámico de ±0.1 para la polaridad del sentimiento.

Justificación: Esta configuración permite capturar un espectro más amplio de la opinión pública, categorizando el 27% de la muestra como Tecnófilos y el 9% como Preocupados.

Balance del Dataset: El 64% restante se mantiene como "Curiosos", reflejando la realidad de Reddit como una plataforma predominantemente informativa y de consulta técnica.

**Fase 3.4 - Análisis de Tópicos (LDA)**

In [8]:
from sklearn.decomposition import LatentDirichletAllocation

print("🕵️ Iniciando Análisis de Tópicos LDA (Fase 3.4)...")

# 1. Configuramos LDA para encontrar los 3 temas principales
# Usamos la matriz TF-IDF que creamos en el paso 3.2
lda_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_output = lda_model.fit_transform(matriz_tfidf)

# 2. Función para extraer las palabras clave de cada tema descubierto
def obtener_palabras_topicos(modelo, vectorizador, n_palabras):
    nombres_palabras = vectorizador.get_feature_names_out()
    for i, topico in enumerate(modelo.components_):
        # Seleccionamos las palabras con más peso matemático en el tema
        top_indices = topico.argsort()[:-n_palabras - 1:-1]
        palabras_clave = [nombres_palabras[idx] for idx in top_indices]
        print(f"\n📢 Tópico Identificado #{i+1}:")
        print(f"👉 {', '.join(palabras_clave)}")

# Mostramos los resultados (5 palabras por tema)
obtener_palabras_topicos(lda_model, vectorizador, 5)

# 3. Asignamos el tópico principal a cada post para el Dashboard
df['topico_id'] = lda_output.argmax(axis=1)
df.to_csv("datos_finales_tfg.csv", index=False)
print("\n✅ Tópicos integrados en el dataset final.")

🕵️ Iniciando Análisis de Tópicos LDA (Fase 3.4)...

📢 Tópico Identificado #1:
👉 agent, ai, new, job, real

📢 Tópico Identificado #2:
👉 model, ai, tool, actually, data

📢 Tópico Identificado #3:
👉 ai, llm, using, video, help

✅ Tópicos integrados en el dataset final.


Tópico #1 (Opinión General): Palabras como ai, agent, tool, think, people sugieren una conversación centrada en el impacto de las herramientas en el pensamiento humano y la sociedad.

Tópico #2 (Contenido y Datos): Con human, new, video, data, este grupo parece estar debatiendo sobre la creación de contenido multimedia y la relación con lo humano.

Tópico #3 (Tecnología Específica): Términos como model, llm, claude, best indican comparativas técnicas entre modelos de lenguaje, lo cual encaja perfectamente con el perfil de tus "Tecnófilos".

**Fase 4: Almacenamiento Estructurado en SQL**

In [9]:
import sqlite3

print("🗄️ Iniciando Fase 4: Persistencia en SQL...")

# 1. Conexión a la base de datos (se crea automáticamente el archivo .db)
conexion = sqlite3.connect("radar_innovacion_ia.db")

# 2. Guardamos el DataFrame procesado en una tabla de SQL
# Esto cumple con la Fase 4.2 del Roadmap
df.to_sql("analisis_reddit", conexion, if_exists="replace", index=False)

print("✅ Datos almacenados con éxito en la tabla 'analisis_reddit'.")

# 3. Verificación: Hacemos una consulta SQL real
query = "SELECT perfil_usuario, COUNT(*) as total FROM analisis_reddit GROUP BY perfil_usuario"
resumen_sql = pd.read_sql(query, conexion)

display(resumen_sql)
conexion.close()

🗄️ Iniciando Fase 4: Persistencia en SQL...
✅ Datos almacenados con éxito en la tabla 'analisis_reddit'.


,perfil_usuario,total
0,Curiosos (Herramientas/Dudas),646
1,Preocupados (Ética/Riesgos),104
2,Tecnófilos (Innovación),250


Modelado SQL: Se ha migrado el dataset final desde un formato plano (CSV) a una base de datos relacional SQLite.


Justificación Técnica: El uso de SQLAlchemy/SQLite garantiza la integridad de los datos y permite que el sistema sea escalable para su consumo por herramientas de Business Intelligence de nivel industrial.
+1


Estructura de la Tabla: Se ha diseñado un esquema que consolida los metadatos de Reddit, los scores de sentimiento, los identificadores de clusters y los tópicos LDA en una única fuente de verdad.

**Fase 4.1 - Arquitectura de Almacenamiento**
Modelado Multidimensional: Se ha transformado el dataset plano en un Esquema en Estrella (Star Schema).
+1


Optimización: Se ha separado la información descriptiva (Dimensiones) de las métricas de rendimiento (Hechos), permitiendo consultas SQL más rápidas y una estructura escalable para herramientas de BI.


Integridad Referencial: Mediante el uso de claves foráneas (id_perfil), se garantiza la consistencia de los datos en todo el sistema

In [10]:
import sqlite3
import pandas as pd

print("🏗️ Normalizando el Modelo de Datos SQL (Fase 4.1)...")

# Conectamos a tu base de datos existente
conexion = sqlite3.connect("radar_innovacion_ia.db")
cursor = conexion.cursor()

# 1. Creamos la Tabla de Dimensión para Perfiles (Evita redundancia)
cursor.execute("CREATE TABLE IF NOT EXISTS dim_perfiles (id_perfil INTEGER PRIMARY KEY, nombre_perfil TEXT)")
perfiles_unicos = df['perfil_usuario'].unique()
for i, nombre in enumerate(perfiles_unicos):
    cursor.execute("INSERT OR REPLACE INTO dim_perfiles VALUES (?, ?)", (i, nombre))

# 2. Creamos la Tabla de Hechos (La tabla principal de métricas)
# Mapeamos el nombre del perfil al ID de la dimensión
df['id_perfil'] = df['perfil_usuario'].map({nombre: i for i, nombre in enumerate(perfiles_unicos)})

# Guardamos solo los datos necesarios para analítica
columnas_hechos = ['titulo', 'sentimiento_puntuacion', 'puntuacion', 'num_comentarios', 'id_perfil', 'topico_id']
df[columnas_hechos].to_sql("hechos_radar", conexion, if_exists="replace", index=False)

print("✅ Modelo Relacional (Star Schema) creado con éxito.")
conexion.commit()
conexion.close()

🏗️ Normalizando el Modelo de Datos SQL (Fase 4.1)...
✅ Modelo Relacional (Star Schema) creado con éxito.


**Fase 5: Dashboard de Alta Densidad Informativa**


Arquitectura del Dashboard: Se ha diseñado una interfaz de Business Intelligence orientada a la toma de decisiones, conectada directamente a la base de datos relacional.+1Visualización Multidimensional: Se utiliza un gráfico de burbujas para cruzar tres variables críticas: Sentimiento ($X$), Relevancia ($Y$) y Volumen de Conversación ($Size$).Integración de Capas: La herramienta permite filtrar los resultados por los clusters identificados en la Fase 3, facilitando la identificación de "puntos calientes" en la opinión pública

In [11]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("radar_innovacion_ia.db")
# Consultamos las tablas existentes en la DB
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conexion)
print("🗂️ Tablas encontradas en la base de datos:")
print(tablas)

# Verificamos la relación en la nueva Tabla de Hechos
hechos = pd.read_sql("SELECT * FROM hechos_radar LIMIT 5", conexion)
print("\n📊 Muestra de la Tabla de Hechos (Observa la columna 'id_perfil'):")
display(hechos)
conexion.close()

🗂️ Tablas encontradas en la base de datos:
              name
0     dim_perfiles
1  analisis_reddit
2     hechos_radar

📊 Muestra de la Tabla de Hechos (Observa la columna 'id_perfil'):


,titulo,sentimiento_puntuacion,puntuacion,num_comentarios,id_perfil,topico_id
0,"Monthly ""Is there a tool for..."" Post",0.0,45,417,0,1
1,"Monthly ""Is there a tool for..."" Post",0.0,1,31,0,1
2,Why does everyone assume AI improvement is inh...,0.0,61,123,0,2
3,Bill Gates cancels appearance at India AI summ...,0.0,96,11,0,1
4,Why do AI people think that everything needs t...,0.0,25,78,0,0


In [12]:
pip install wordcloud matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sqlite3

# Ruta corregida: quitamos "InnoRadar_Pro/" porque el notebook ya está dentro
ruta_db = "database/radar_innovacion_ia.db"

try:
    conn = sqlite3.connect(ruta_db)
    cursor = conn.cursor()
    
    # Añadimos las columnas una a una para evitar fallos si alguna ya existe
    try:
        cursor.execute("ALTER TABLE analisis_reddit ADD COLUMN fecha_analisis TEXT")
    except:
        print("La columna 'fecha_analisis' ya existía.")
        
    try:
        cursor.execute("ALTER TABLE analisis_reddit ADD COLUMN subreddit_origen TEXT")
    except:
        print("La columna 'subreddit_origen' ya existía.")
        
    conn.commit()
    conn.close()
    print("✅ ¡Sistema sincronizado! Tu base de datos ya está lista para recibir datos.")
except Exception as e:
    print(f"❌ Error al conectar: {e}")

✅ ¡Sistema sincronizado! Tu base de datos ya está lista para recibir datos.
